# PlatformCare: customer impact and incident walkthrough

Read-only exploration of the local synthetic pipeline history. No model calls, cloud access, or optional service calls. Run the healthy, invalid_input, and recovery scenarios using the README first. The cells use only Python standard-library modules. Jupyter itself is optional and must already be installed.


In [ ]:
import json, os, sqlite3
from pathlib import Path
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in roots if (p / "platformcare" / "pipeline.py").exists())
db = Path(os.getenv("PLATFORM_DB", str(ROOT / "data" / "platform.db"))).resolve()
if not db.is_file():
    raise FileNotFoundError("Create synthetic runs using the README first")
def query(sql):
    connection = sqlite3.connect(db.as_uri() + "?mode=ro", uri=True)
    connection.row_factory = sqlite3.Row
    try:
        return [dict(row) for row in connection.execute(sql)]
    finally:
        connection.close()


## Is the customer data ready?

A successful historical output and a healthy latest job are separate signals. The lab freshness objective is 300 seconds. Metrics reflect the stored sample, not production traffic.


In [ ]:
import time
latest = query("SELECT id, status, scenario, started, finished FROM runs ORDER BY started DESC LIMIT 1")
good = query("SELECT finished FROM runs WHERE status='success' ORDER BY finished DESC LIMIT 1")
age = max(0, time.time() - good[0]["finished"]) if good else None
print(json.dumps({"consumer": "Daily usage dashboard", "owner": "demo-data-team",
    "latest_run": latest[0] if latest else None, "output_age_seconds": age,
    "freshness_objective_seconds": 300, "output_within_objective": age is not None and age <= 300}, indent=2))


## What failed, and what should support inspect?

Use the exact failed run ID to connect the error to the validation or dependency runbook. This is SQL-based exploration, not an LLM diagnosis.


In [ ]:
failures = query("SELECT r.id, r.scenario, l.code, l.message FROM runs r JOIN logs l ON l.run_id=r.id WHERE r.status='failed' AND l.level='ERROR' ORDER BY r.started DESC LIMIT 5")
print(json.dumps(failures, indent=2))
print((ROOT / "runbooks" / "validation.md").read_text())


## Support and incident metrics

Display counts alongside averages. Resolution latency includes only resolved incidents. A small demo sample cannot establish reduced support toil or production MTTR.


In [ ]:
print(json.dumps(query("SELECT COUNT(*) AS requests, SUM(CASE WHEN answered IS NULL THEN 1 ELSE 0 END) AS unanswered, AVG(answered-opened) AS mean_first_response_seconds, COUNT(helpful) AS feedback_count, AVG(helpful) AS helpful_fraction FROM requests"), indent=2))
print(json.dumps(query("SELECT COUNT(*) AS incidents, SUM(CASE WHEN resolved IS NULL THEN 1 ELSE 0 END) AS open_incidents, AVG(resolved-opened) AS mean_resolved_duration_seconds FROM incidents"), indent=2))


## Explain the design

Why does API availability not guarantee fresh data? What evidence supports the diagnosis? What would you ask an SRE to review? See `docs/CUSTOMER_METRICS.md` and `docs/AI_CODE_REVIEW.md`. Optional engine/vendor adapters are described in `docs/OPTIONAL_SERVICES.md`; no cell here claims those services ran.
